In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"

silver_table = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql import functions as F

In [0]:
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName").alias("race_name"),
    F.col("date").alias("race_date"),
    F.col("circuitId").alias("circuit_id"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

In [0]:
races_valid_df = races_selected_df.filter(
    F.col("season").isNotNull() & F.col("round").isNotNull()
)

In [0]:
races_distinct_df = races_valid_df.dropDuplicates(["season","round"])

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn("race_name", F.initcap("race_name"))
    )

In [0]:
(
    races_final_df.write
    .format('delta')
    .mode("overwrite")
    .saveAsTable(silver_table)
)